# Simulador de Interferencia Entre Símbolos (ISI)
Mueve el slider para ver cómo un eco en el canal puede generar bits falsos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from ipywidgets import interact, FloatSlider, Layout

# 1. Configuración de la señal (2 Bits)
t = np.linspace(0, 1.2, 1200)
dt = t[1] - t[0]
x = np.where(((t > 0.1) & (t < 0.2)) | ((t > 0.4) & (t < 0.5)), 1.0, 0.0)
noise_threshold = 0.3

def sim_isi(tau):
    h = np.zeros_like(t)
    h[0] = 1.0
    idx_eco = np.abs(t - tau).argmin()
    h[idx_eco] = 0.8
    y = np.convolve(x, h, mode='full')[:len(t)] * dt
    if np.max(y) > 0:
        y = y / np.max(y)
    pks, _ = find_peaks(y, height=noise_threshold, distance=50)
    return h, y, len(pks)

def actualizar_grafico(retardo=0.15):
    h, y, n_eventos = sim_isi(retardo)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    plt.subplots_adjust(hspace=0.4)

    ax1.plot(t, x, 'b', linewidth=2)
    ax1.set_title("1. Señal Transmitida (2 Bits)", fontweight='bold')
    ax1.grid(True, alpha=0.3)

    ax2.stem(t[::10], h[::10], basefmt=" ", linefmt='g', markerfmt='go')
    ax2.set_title(f"2. Canal h(t) - Eco en {retardo:.2f}s", fontweight='bold')
    ax2.grid(True, alpha=0.3)

    color_res = 'g' if n_eventos == 2 else 'r'
    ax3.plot(t, y, color=color_res, linewidth=2)
    ax3.axhline(y=noise_threshold, color='gray', linestyle='--')
    ax3.set_title(f"3. Recibido: {n_eventos} eventos detectados", color=color_res, fontweight='bold')
    ax3.set_xlabel("Tiempo (s)")
    ax3.grid(True, alpha=0.3)
    plt.show()

interact(actualizar_grafico, 
         retardo=FloatSlider(value=0.15, min=0.0, max=0.6, step=0.01, 
                             description='Retardo:', layout=Layout(width='70%')));